# 02 · Propiedades Estelares y Componentes del BCG+ICL
**Metodología:** §4, §5, §6 de Mayes+2026

Clasificamos las partículas estelares usando los catálogos de ensamblaje estelar
(Rodriguez-Gomez+2016, disponibles en el cluster):
- **In situ** (canal 0): formadas en el subhalo central
- **Mergers completados** (canal 1): de galaxias totalmente disruptadas
- **Stripped / Sobrevivientes** (canal 2): de galaxias aún existentes

Luego estudiamos **metalicidad**, **color B−V** y **edad** del ICL y BCG
con sus gradientes radiales. Equivalente a **Figs. 7–12** de Mayes+2026.


In [1]:
import sys, os, pickle
import numpy as np
import h5py
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, './original_shift_code')
import illustris_python as il
import Catalogue
import params_icl as P

# Carpetas de salida para las figuras de este notebook
FIG_PDF = './figuras/02_propiedades_estelares/pdf'
FIG_PNG = './figuras/02_propiedades_estelares/png'
os.makedirs(FIG_PDF, exist_ok=True)
os.makedirs(FIG_PNG, exist_ok=True)

%matplotlib inline
plt.rcParams.update({'figure.dpi': 110, 'font.size': 12,
                     'axes.spines.top': False, 'axes.spines.right': False})

cosmo = FlatLambdaCDM(H0=67.74, Om0=0.3089)
Header = il.groupcat.loadHeader(P.basePath, P.SNAP)

with h5py.File(P.CATALOG_OUT, 'r') as f:
    group_idx   = f['group_idx'][:]
    M200c       = f['M200c_Msun'][:]
    R200c       = f['R200c_kpc'][:]
    GroupPos    = f['GroupPos_kpc'][:]
    bcg_sub_idx = f['bcg_sub_idx'][:]
    icl_frac    = f['icl_frac'][:]
    dyn_state   = f['dyn_state'][:] if 'dyn_state' in f else None

n_cl = len(group_idx)
lM   = np.log10(M200c)
COLORS_COMP = {0: '#E91E63', 1: '#9C27B0', 2: '#00BCD4'}
LABELS_COMP = {0: 'In situ', 1: 'Mergers completados', 2: 'Stripped (sobrev.)'}
print(f"Catálogo cargado: {n_cl} cúmulos")

KeyError: "Unable to synchronously open object (object 'icl_frac' doesn't exist)"

## §2.3 – Catálogos de ensamblaje estelar (Rodriguez-Gomez+2016)

Ruta en el cluster: `P.SA_FILE`

**Estructura real del catálogo** (estadísticas AGREGADAS por subhalo, NO por partícula):
```
f['Snapshot_99']['StellarMassInSitu'][sub_idx]               # masa formada in situ
f['Snapshot_99']['StellarMassExSitu'][sub_idx]               # masa total ex situ
f['Snapshot_99']['StellarMassFromCompletedMergers'][sub_idx] # de fusiones terminadas
f['Snapshot_99']['StellarMassFromCompletedMergersMajor'][sub_idx]      # solo fusiones mayores
f['Snapshot_99']['StellarMassFromCompletedMergersMajorMinor'][sub_idx] # mayor + menores
f['Snapshot_99']['StellarMassFromOngoingMergers'][sub_idx]   # de satélites aún existentes
f['Snapshot_99']['StellarMassFromOngoingMergersMajor'][sub_idx]
f['Snapshot_99']['StellarMassTotal'][sub_idx]                # masa total (check)
```
Unidades: 1×10¹⁰ M☉/h (como los datos internos de TNG).

**Nota importante**: este catálogo da fracciones totales del subhalo central (= BCG+ICL juntos
en la definición SUBFIND). Para separar componentes por región espacial (BCG vs ICL)
sería necesario un catálogo SA por-partícula, que no está disponible.


In [ ]:
# Carga estadísticas SA agregadas por subhalo (un valor por subhalo en snap 99)
with h5py.File(P.SA_FILE, 'r') as f:
    s99 = f['Snapshot_99']
    sa_insitu    = s99['StellarMassInSitu'][:]                    * P.UM
    sa_exsitu    = s99['StellarMassExSitu'][:]                    * P.UM
    sa_mergers   = s99['StellarMassFromCompletedMergers'][:]      * P.UM
    sa_maj_comp  = s99['StellarMassFromCompletedMergersMajor'][:] * P.UM
    sa_ongoing   = s99['StellarMassFromOngoingMergers'][:]        * P.UM
    sa_maj_ong   = s99['StellarMassFromOngoingMergersMajor'][:]   * P.UM
    sa_total     = s99['StellarMassTotal'][:]                     * P.UM

print(f"N subhalos en catálogo SA: {len(sa_total):,}")
# Verificar con primer BCG
i0 = 0; si = bcg_sub_idx[i0]
print(f"\nBCG #0 (sub_idx={si}):")
print(f"  M_insitu  = {sa_insitu[si]:.3e} M☉   f = {sa_insitu[si]/sa_total[si]:.3f}")
print(f"  M_exsitu  = {sa_exsitu[si]:.3e} M☉   f = {sa_exsitu[si]/sa_total[si]:.3f}")
print(f"  M_mergers = {sa_mergers[si]:.3e} M☉  f = {sa_mergers[si]/sa_total[si]:.3f}")
print(f"  M_ongoing = {sa_ongoing[si]:.3e} M☉  f = {sa_ongoing[si]/sa_total[si]:.3f}")
print(f"  M_total   = {sa_total[si]:.3e} M☉")

# Fracciones para todos los BCGs
sa_tot_bcg = sa_total[bcg_sub_idx]
mask_ok    = sa_tot_bcg > 0
f_insitu  = np.where(mask_ok, sa_insitu[bcg_sub_idx]   / sa_tot_bcg, np.nan)
f_exsitu  = np.where(mask_ok, sa_exsitu[bcg_sub_idx]   / sa_tot_bcg, np.nan)
f_mergers = np.where(mask_ok, sa_mergers[bcg_sub_idx]  / sa_tot_bcg, np.nan)
f_ongoing = np.where(mask_ok, sa_ongoing[bcg_sub_idx]  / sa_tot_bcg, np.nan)
f_maj_comp= np.where(mask_ok, sa_maj_comp[bcg_sub_idx] / sa_tot_bcg, np.nan)
f_maj_ong = np.where(mask_ok, sa_maj_ong[bcg_sub_idx]  / sa_tot_bcg, np.nan)

print(f"\nMedianas (BCG+ICL total):")
print(f"  f_insitu  = {np.nanmedian(f_insitu):.3f}")
print(f"  f_mergers = {np.nanmedian(f_mergers):.3f}  (completados)")
print(f"  f_ongoing = {np.nanmedian(f_ongoing):.3f}  (stripped / sobrevivientes)")
print(f"  f_exsitu  = {np.nanmedian(f_exsitu):.3f}  (total ex-situ)")


## Función principal: cargar y clasificar partículas de un cúmulo

In [ ]:
# Reutilizamos rotate_by_inertia_tensor del notebook 01
# (reproducida aquí para que este notebook sea autosuficiente)

from scipy.interpolate import interp1d

def rotate_by_inertia_tensor(pos_rel, mass, r_lim=np.inf):
    dist = np.linalg.norm(pos_rel, axis=1)
    ok   = (dist > 0) & (dist <= r_lim) & np.isfinite(mass)
    p, m = pos_rel[ok], mass[ok]
    if m.sum() == 0 or len(m) < 4:
        return pos_rel, np.eye(3)
    w    = 1.0 / dist[ok]**2
    mtot = np.sum(m)
    Ixx  = np.sum(m * p[:,0]**2 * w) / mtot
    Iyy  = np.sum(m * p[:,1]**2 * w) / mtot
    Izz  = np.sum(m * p[:,2]**2 * w) / mtot
    Ixy  = np.sum(m * p[:,0] * p[:,1] * w) / mtot
    Ixz  = np.sum(m * p[:,0] * p[:,2] * w) / mtot
    Iyz  = np.sum(m * p[:,1] * p[:,2] * w) / mtot
    I = np.array([[Ixx,Ixy,Ixz],[Ixy,Iyy,Iyz],[Ixz,Iyz,Izz]])
    ev, evec = np.linalg.eigh(I)
    R_mat = evec[:, np.argsort(ev)[::-1]].T
    return pos_rel @ R_mat.T, R_mat

def sb_profile_r(r_2d, lum_r, r_max_kpc, n_bins=60):
    r_bins = np.logspace(np.log10(0.5), np.log10(r_max_kpc), n_bins+1)
    r_mid  = np.sqrt(r_bins[:-1]*r_bins[1:])
    mu_r   = np.full(n_bins, np.nan)
    for k,(r1,r2) in enumerate(zip(r_bins[:-1],r_bins[1:])):
        mk = (r_2d>=r1)&(r_2d<r2)
        if mk.sum()==0: continue
        sl = lum_r[mk].sum() / (np.pi*((r2*1e3)**2-(r1*1e3)**2))
        if sl>0: mu_r[k] = P.SB_CONST - 2.5*np.log10(sl)
    return r_mid, mu_r

def holmberg_radius(r_mid, mu_r, mu_cut=P.MU_HOLMBERG):
    valid = np.isfinite(mu_r)&(r_mid>0)
    if valid.sum()<3: return np.nan
    rv,mv = r_mid[valid],mu_r[valid]
    idx   = np.argsort(rv)
    rv,mv = rv[idx],mv[idx]
    if mv[0]>mu_cut or mv[-1]<mu_cut: return np.nan
    try:
        r_h = float(interp1d(mv,rv,fill_value='extrapolate')(mu_cut))
        return r_h if 0<r_h<=rv[-1]*1.2 else np.nan
    except: return np.nan

def hmr(r, m):
    """Radio de semi-masa (Half-Mass Radius)."""
    if len(r)==0 or m.sum()==0: return np.nan
    idx = np.argsort(r)
    cum = np.cumsum(m[idx])
    i50 = np.searchsorted(cum, cum[-1]/2)
    return r[idx[min(i50, len(r)-1)]]

def load_cluster_stars(sub_id, cen_pos, header=Header):
    """
    Carga partículas estelares del subhalo central, las rota y separa BCG/ICL.
    No requiere catálogo SA por-partícula.
    """
    fields = ['Coordinates','Masses',
              'GFM_StellarPhotometrics','GFM_Metallicity','GFM_StellarFormationTime']
    st = il.snapshot.loadSubhalo(P.basePath, P.SNAP, int(sub_id), 'stars', fields=fields)

    pos   = Catalogue.Distance_3D(st['Coordinates']*P.UL, cen_pos, header['BoxSize']*P.UL)
    mass  = st['Masses'] * P.UM
    phot  = st['GFM_StellarPhotometrics']
    metal = st['GFM_Metallicity']
    aform = st['GFM_StellarFormationTime']

    pos_rot, _ = rotate_by_inertia_tensor(pos, mass)
    r_2d  = np.sqrt(pos_rot[:,0]**2 + pos_rot[:,1]**2)
    lum_r = 10**((P.M_SUN_R_AB - phot[:,5]) / 2.5)

    r_m, mu = sb_profile_r(r_2d, lum_r, np.percentile(r_2d, 99))
    r_h = holmberg_radius(r_m, mu)
    mask_bcg = r_2d <= r_h if np.isfinite(r_h) else np.zeros(len(r_2d), bool)
    mask_icl = ~mask_bcg

    return {'mass':mass, 'phot':phot, 'metal':metal, 'aform':aform,
            'r_2d':r_2d, 'pos_rot':pos_rot, 'lum_r':lum_r,
            'r_holmberg':r_h, 'mask_bcg':mask_bcg, 'mask_icl':mask_icl}


## §4 – Fracciones de masa por componente y HMR (equivalente a Figs. 7, 8, 9)

Las fracciones de componentes se leen directamente del catálogo SA agregado (ya calculadas arriba).
Para las propiedades espaciales (HMR, radios de semi-masa) se cargan las partículas.

**Nota**: sin catálogo SA por-partícula no es posible separar los componentes (in situ /
mergers / stripped) dentro del BCG vs dentro del ICL. Las fracciones de componentes
corresponden siempre al subhalo central completo (BCG+ICL juntos).


In [ ]:
def linfit(x, y):
    ok = np.isfinite(x) & np.isfinite(y)
    if ok.sum() < 3: return np.nan, np.nan
    sl, ic, *_ = linregress(x[ok], y[ok])
    return sl, ic

# ── Figura 8: fracciones de componentes BCG+ICL del catálogo SA ──────────
COLORS_COMP = {'In situ':'#E91E63', 'Mergers':'#9C27B0', 'Stripped':'#00BCD4'}

fig, ax = plt.subplots(figsize=(8,6))
for f_arr, color, label in [
    (f_insitu,  '#E91E63', 'In situ'),
    (f_mergers, '#9C27B0', 'Mergers completados'),
    (f_ongoing, '#00BCD4', 'Stripped (sobrev.)'),
]:
    ok = np.isfinite(f_arr)
    ax.scatter(lM[ok], f_arr[ok], color=color, label=label, s=20, alpha=0.7)
    sl, ic = linfit(lM[ok], f_arr[ok])
    if np.isfinite(sl):
        xx = np.linspace(lM.min(), lM.max(), 100)
        ax.plot(xx, sl*xx+ic, color=color, lw=1.8, ls='--', label=f'β={sl:.3f}')

ax.set_xlabel(r'$\log M_{{200c}}\;[M_\odot]$')
ax.set_ylabel('Fracción de masa estelar')
ax.set_title('Componentes del BCG+ICL – catálogo SA (Fig. 8)')
ax.legend(fontsize=9, ncol=2)
plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig08_componentes_bcgicl.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig08_componentes_bcgicl.png', bbox_inches='tight', dpi=150)
plt.show()

print(f"Medianas  f_insitu={np.nanmedian(f_insitu):.3f}  "
      f"f_mergers={np.nanmedian(f_mergers):.3f}  "
      f"f_ongoing={np.nanmedian(f_ongoing):.3f}")

In [ ]:
# ── Figura 8b: desglose mayor vs menor en mergers ────────────────────────
f_min_comp = np.where(mask_ok,
    (sa_mergers[bcg_sub_idx] - sa_maj_comp[bcg_sub_idx]) / sa_tot_bcg, np.nan)
f_min_ong  = np.where(mask_ok,
    (sa_ongoing[bcg_sub_idx] - sa_maj_ong[bcg_sub_idx])  / sa_tot_bcg, np.nan)

fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax, f_maj, f_min, title in [
    (axes[0], f_maj_comp, f_min_comp, 'Mergers completados'),
    (axes[1], f_maj_ong,  f_min_ong,  'Stripped (ongoing)'),
]:
    for f_arr, col, lbl in [(f_maj,'#9C27B0','Mayores (>1:4)'),
                             (f_min,'#CE93D8','Menores (<1:4)')]:
        ok = np.isfinite(f_arr)
        ax.scatter(lM[ok], f_arr[ok], color=col, label=lbl, s=18, alpha=0.7)
        sl, ic = linfit(lM[ok], f_arr[ok])
        if np.isfinite(sl):
            xx = np.linspace(lM.min(), lM.max(), 100)
            ax.plot(xx, sl*xx+ic, color=col, lw=1.5, ls='--', label=f'β={sl:.3f}')
    ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel('Fracción de masa')
    ax.set_title(f'{title}: mayor vs menor'); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig08b_mayor_menor.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig08b_mayor_menor.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── HMR para BCG e ICL (Holmberg) usando partículas ─────────────────────
# (no por componente SA, sino por región espacial)
print("Calculando HMR por región espacial (BCG / ICL)...")
hmr_bcg = np.full(n_cl, np.nan)
hmr_icl = np.full(n_cl, np.nan)
m_bcg   = np.full(n_cl, np.nan)
m_icl   = np.full(n_cl, np.nan)

for i in range(n_cl):
    try: d = load_cluster_stars(bcg_sub_idx[i], GroupPos[i])
    except: continue
    r_h = d['r_holmberg']
    if not np.isfinite(r_h): continue
    for mask, hmr_arr, m_arr in [(d['mask_bcg'], hmr_bcg, m_bcg),
                                  (d['mask_icl'], hmr_icl, m_icl)]:
        r_k = d['r_2d'][mask]; ms_k = d['mass'][mask]
        m_arr[i] = ms_k.sum()
        if len(r_k) > 0 and ms_k.sum() > 0:
            idx = np.argsort(r_k); cum = np.cumsum(ms_k[idx])
            i50 = np.searchsorted(cum, cum[-1]/2)
            hmr_arr[i] = r_k[idx[min(i50, len(r_k)-1)]]
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}", end="\r")
print("\nListo.")

fig, axes = plt.subplots(1,2,figsize=(13,5))
for ax, arr, title in [(axes[0], hmr_bcg, 'BCG'), (axes[1], hmr_icl, 'ICL')]:
    ok = np.isfinite(arr) & (arr>0)
    ax.scatter(lM[ok], arr[ok], s=20, alpha=0.7, color='steelblue')
    sl, ic = linfit(lM[ok], np.log10(arr[ok]))
    if np.isfinite(sl):
        xx = np.linspace(lM.min(), lM.max(), 100)
        ax.plot(xx, 10**(sl*xx+ic), 'k--', lw=1.5, label=f'β={sl:.3f}')
    ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel('HMR [kpc]')
    ax.set_yscale('log'); ax.set_title(f'Radio de semi-masa {title} (Fig. 7)')
    ax.legend()
plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig07_hmr_bcg_icl.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig07_hmr_bcg_icl.png', bbox_inches='tight', dpi=150)
plt.show()

## §6.1 – Metalicidad (equivalente a Fig. 10 del paper)

In [ ]:
print("Calculando propiedades estelares para todos los cúmulos...")

def lookback(aform):
    z = np.clip(1/np.where(aform>0,aform,1e-4) - 1, 0, 30)
    return cosmo.lookback_time(z).value

mean_Z_icl   = np.full(n_cl, np.nan)
mean_Z_bcg   = np.full(n_cl, np.nan)
mean_BV_icl  = np.full(n_cl, np.nan)
mean_BV_bcg  = np.full(n_cl, np.nan)
mean_age_icl = np.full(n_cl, np.nan)
mean_age_bcg = np.full(n_cl, np.nan)
grad_Z       = np.full(n_cl, np.nan)   # gradiente metalicidad BCG+ICL
grad_BV      = np.full(n_cl, np.nan)   # gradiente color BCG+ICL

for i in range(n_cl):
    try: d = load_cluster_stars(bcg_sub_idx[i], GroupPos[i])
    except: continue

    for mask, Z_arr, BV_arr, age_arr in [
        (d['mask_icl'], mean_Z_icl,  mean_BV_icl,  mean_age_icl),
        (d['mask_bcg'], mean_Z_bcg,  mean_BV_bcg,  mean_age_bcg),
    ]:
        if mask.sum()==0: continue
        Z_arr[i]   = np.log10(np.average(d['metal'][mask], weights=d['mass'][mask])
                               / P.Z_SUN)
        lr_k       = 10**((P.M_SUN_R_AB - d['phot'][mask,5]) / 2.5)
        BV_arr[i]  = np.average(d['phot'][mask,1] - d['phot'][mask,2], weights=lr_k)
        age_arr[i] = np.average(lookback(d['aform'][mask]), weights=d['mass'][mask])

    # Gradiente radial de metalicidad (BCG+ICL completo)
    r2 = d['r_2d']
    rg = np.logspace(np.log10(max(r2.min(),0.5)), np.log10(np.percentile(r2,99)), 21)
    rc = np.sqrt(rg[:-1]*rg[1:])
    Z_r = np.array([np.log10(np.average(d['metal'][(r2>=r1)&(r2<r2_)],
                                          weights=d['mass'][(r2>=r1)&(r2<r2_)])/P.Z_SUN)
                     if ((r2>=r1)&(r2<r2_)).sum()>0 else np.nan
                     for r1,r2_ in zip(rg[:-1],rg[1:])])
    BV_r= np.array([np.average(d['phot'][(r2>=r1)&(r2<r2_),1]-d['phot'][(r2>=r1)&(r2<r2_),2],
                                 weights=10**((P.M_SUN_R_AB-d['phot'][(r2>=r1)&(r2<r2_),5])/2.5))
                     if ((r2>=r1)&(r2<r2_)).sum()>0 else np.nan
                     for r1,r2_ in zip(rg[:-1],rg[1:])])
    ok_Z  = np.isfinite(Z_r) & np.isfinite(np.log10(rc))
    ok_BV = np.isfinite(BV_r)& np.isfinite(np.log10(rc))
    if ok_Z.sum()>=3:
        sl,*_ = linregress(np.log10(rc[ok_Z]), Z_r[ok_Z]); grad_Z[i]=sl
    if ok_BV.sum()>=3:
        sl,*_ = linregress(np.log10(rc[ok_BV]), BV_r[ok_BV]); grad_BV[i]=sl
    if (i+1)%10==0: print(f"  {i+1}/{n_cl}", end="\r")
print("\nListo.")


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(13,5))

# Panel izq: metalicidad media
ax = axes[0]
for arr, col, lbl in [(mean_Z_icl,'royalblue','ICL'),(mean_Z_bcg,'tomato','BCG')]:
    ok = np.isfinite(arr)
    ax.scatter(lM[ok], arr[ok], color=col, s=20, alpha=0.7, label=lbl)
    sl,ic = linfit(lM[ok], arr[ok])
    xx = np.linspace(lM.min(), lM.max(), 100)
    ax.plot(xx, sl*xx+ic, color=col, lw=1.8, ls='--')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel(r'$\langle\log Z_*/Z_\odot\rangle$')
ax.set_title('Metalicidad media ICL vs BCG (Fig. 10 top)'); ax.legend()

# Panel der: gradiente de metalicidad
ax = axes[1]
ok = np.isfinite(grad_Z)
ax.scatter(lM[ok], grad_Z[ok], color='slategray', s=20, alpha=0.7)
ax.axhline(0, color='k', lw=1, ls='--')
sl,ic = linfit(lM[ok], grad_Z[ok])
xx = np.linspace(lM.min(), lM.max(), 100)
ax.plot(xx, sl*xx+ic, 'k-', lw=1.8, label=f'β={sl:.3f}')
ax.set_xlabel(r'$\log M_{{200c}}$')
ax.set_ylabel(r'Gradiente $d\log Z / d\log r$')
ax.set_title('Gradiente metalicidad (Fig. 10 bottom)'); ax.legend()

plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig10_metalicidad.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig10_metalicidad.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"Todos los gradientes negativos: {(grad_Z[ok]<0).all()}")

### §6.2 – Color B−V (Fig. 11) · §6.3 – Edades (Fig. 12)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(16,5))

# ── B−V medio ──────────────────────────────────────────────────────────────
ax = axes[0]
for arr,col,lbl in [(mean_BV_icl,'royalblue','ICL'),(mean_BV_bcg,'tomato','BCG')]:
    ok = np.isfinite(arr)
    ax.scatter(lM[ok],arr[ok],color=col,s=20,alpha=0.7,label=lbl)
    sl,ic = linfit(lM[ok],arr[ok])
    ax.plot(np.linspace(lM.min(),lM.max(),100),
            sl*np.linspace(lM.min(),lM.max(),100)+ic,color=col,lw=1.8,ls='--')
ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel(r'$\langle B-V\rangle$')
ax.set_title('Color B−V medio (Fig. 11 top)'); ax.legend()

# ── Gradiente de color ─────────────────────────────────────────────────────
ax = axes[1]
ok = np.isfinite(grad_BV)
ax.scatter(lM[ok],grad_BV[ok],color='slategray',s=20,alpha=0.7)
ax.axhline(0,color='k',lw=1,ls='--')
sl,ic = linfit(lM[ok],grad_BV[ok])
ax.plot(np.linspace(lM.min(),lM.max(),100),
        sl*np.linspace(lM.min(),lM.max(),100)+ic,'k-',lw=1.8,label=f'β={sl:.3f}')
ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel(r'Gradiente $d(B-V)/d\log r$')
ax.set_title('Gradiente color (Fig. 11 bottom)'); ax.legend()

# ── Edad estelar ──────────────────────────────────────────────────────────
ax = axes[2]
for arr,col,lbl in [(mean_age_icl,'royalblue','ICL'),(mean_age_bcg,'tomato','BCG')]:
    ok = np.isfinite(arr)
    ax.scatter(lM[ok],arr[ok],color=col,s=20,alpha=0.7,label=lbl)
    sl,ic = linfit(lM[ok],arr[ok])
    ax.plot(np.linspace(lM.min(),lM.max(),100),
            sl*np.linspace(lM.min(),lM.max(),100)+ic,color=col,lw=1.8,ls='--')
ax.set_xlabel(r'$\log M_{{200c}}$'); ax.set_ylabel('Edad [Gyr lookback]')
ax.set_title('Edad estelar ICL vs BCG (Fig. 12)'); ax.legend()

plt.tight_layout()
plt.savefig(f'{FIG_PDF}/fig11_12_color_edad.pdf', bbox_inches='tight')
plt.savefig(f'{FIG_PNG}/fig11_12_color_edad.png', bbox_inches='tight', dpi=150)
plt.show()
print(f"ICL más azul que BCG: {np.nanmean(mean_BV_icl)<np.nanmean(mean_BV_bcg)}")
print(f"Δ(B-V) = {np.nanmean(mean_BV_bcg)-np.nanmean(mean_BV_icl):.3f}")